# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: Linear Regression (readable baseline) then Random Forest Regression, both
followed by permutation importance, predicting `ctr_gap`.**

Lane 4 has no observed outcome label (established in `w02_ml_task_framing`), `ctr_gap`
is a proxy I constructed, not something FlyRank measured happening. Classifying against
an invented label built from the same inputs my Week-4 rule already uses would just
relearn my own rule back, that trap is exactly what `w03_data_contract` caught: adding
`ctr` as a "feature" gave a fake R^2 = 1.000, while the honest version (position and
impressions only) gave R^2 = 0.002, real, but close to nothing.

So this week asks a different, still-honest question: **can pages' content and keyword
characteristics, things the baseline rule never looks at, explain any of the CTR gap
the rule measures?** That is the "what drives X" shape from the training-honest-models
table, simple model plus permutation importance, not a classification/precision@K shape,
because there is no label to classify against.

**Same metric as my baseline, made fair:** the baseline never predicts `ctr_gap` from
independent information, it only measures the gap directly from `ctr` and
`position_tier`, the very columns the gap is built from. So its implicit R^2 at this
task is 0 by construction, it uses none of the content/keyword features below. Any R^2
above 0 from a real model is genuinely new information the rule could never have
surfaced.

**Features used (all knowable before any editorial decision, none touch the label's own
ingredients):** `content_type`, `main_intent`, `freshness_tier`, `age_tier`,
`search_volume`, `competition`, `cpc`, `word_count`.

**Deliberately excluded (these ARE the label's ingredients, using them would be the
exact leakage trap from `w03_data_contract`):** `ctr`, `position_tier`,
`tier_avg_ctr`, `avg_position`, `impressions_90d` (used only to define the trustworthy
population below, never as a feature).

**Also excluded:** `provider_used` (71.5% missing) and `model_used` (19.1% missing),
too sparse to trust as features this week, noted here rather than silently dropped.

In [11]:
import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir("flyrank-ml-internship"):
        import subprocess
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/solasobambo-prog/flyrank-ml-internship.git"],
            check=True,
        )
    os.chdir("flyrank-ml-internship")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

label_ingredients = ["ctr", "position_tier", "avg_position", "impressions_90d"]
feature_candidates = ["content_type", "main_intent", "freshness_tier", "age_tier",
                       "search_volume", "competition", "cpc", "word_count"]

print("Label ingredients (excluded from features):", label_ingredients)
print("Feature candidates (independent of the label):", feature_candidates)
print()
print("Confirmed no overlap:", set(label_ingredients).isdisjoint(feature_candidates))
print()
print("Missing % by candidate feature:")
print((df[feature_candidates].isna().mean() * 100).round(1))

Label ingredients (excluded from features): ['ctr', 'position_tier', 'avg_position', 'impressions_90d']
Feature candidates (independent of the label): ['content_type', 'main_intent', 'freshness_tier', 'age_tier', 'search_volume', 'competition', 'cpc', 'word_count']

Confirmed no overlap: True

Missing % by candidate feature:
content_type       0.0
main_intent        7.9
freshness_tier     0.0
age_tier           0.0
search_volume      8.2
competition        8.2
cpc                8.2
word_count        25.7
dtype: float64


## 2. Split design

**Grouped by client, 80/20, `GroupShuffleSplit` on `client_id`, not a random row split.**

Pages from the same client likely share editorial templates, target keyword strategy,
and content production choices (same `provider_used`/`model_used` patterns, similar
`main_intent` mix). A random row split would let the model see some of a client's pages
in training and others from the same client in test, learning that client's quirks
rather than a generalizable content-to-CTR-gap relationship, then getting credit for
"predicting" something it actually memorized. Grouping by client closes that leak.

Also restricted to the same trustworthy floor as the baseline
(`impressions_90d >= 100`), since Signal 2 in ML-07 already showed `ctr_gap` is noisy
below that line, training a model to predict noise would not be honest.

**Honest note:** with only 32 clients total and one client holding 23% of all rows, a
clean 80/20 split by client lands at roughly 16% of rows in test, not exactly 20%,
because whole clients move together. That is expected with this few groups, not a bug.

In [12]:
from sklearn.model_selection import GroupShuffleSplit

# same trustworthy floor as the baseline (Signal 1's floor, ML-07): ctr_gap is only
# meaningful where impressions_90d >= 100
trustworthy = df[df["impressions_90d"] >= 100].copy()
tier_avg_ctr = trustworthy.groupby("position_tier")["ctr"].mean()
trustworthy["tier_avg_ctr"] = trustworthy["position_tier"].map(tier_avg_ctr)
trustworthy["ctr_gap"] = trustworthy["ctr"] - trustworthy["tier_avg_ctr"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(trustworthy, groups=trustworthy["client_id"]))
train_df = trustworthy.iloc[train_idx]
test_df = trustworthy.iloc[test_idx]

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

print(f"Total rows (impressions_90d >= 100): {len(trustworthy):,}")
print(f"Train: {len(train_df):,} rows, {len(train_clients)} clients")
print(f"Test:  {len(test_df):,} rows, {len(test_clients)} clients")
print(f"Clients in both train and test: {len(train_clients & test_clients)}")
print(f"Test set share of rows: {len(test_df)/len(trustworthy):.1%}")


Total rows (impressions_90d >= 100): 22,006
Train: 18,392 rows, 24 clients
Test:  3,614 rows, 6 clients
Clients in both train and test: 0
Test set share of rows: 16.4%


## 3. Train + compare vs my baseline

**Comparison table (same trustworthy population, same client-grouped split as
Section 2, same target: `ctr_gap`):**

| model | R^2 (test) | MAE (test) | note |
|---|---|---|---|
| Baseline rule (ML-07) | 0.0000 | n/a | uses none of these features, by construction |
| DummyRegressor (predict train mean) | -0.0143 | 0.2192 | trivial reference point |
| Linear Regression | -0.5398 | 0.2713 | worse than predicting the mean |
| Random Forest (depth=6) | -1.5655 | 0.3087 | worse still |

**Honest result: neither model beats the baseline, and both do worse than simply
predicting the training mean.** Base rate for scale, the standard deviation of
`ctr_gap` on the test set, is 0.327, so an MAE around 0.27 to 0.31 is not competitive
with just guessing a constant. This is a real negative finding, not a bug, and it is
worth taking seriously rather than tuning until a number looks better.

**What permutation importance says, on the held-out test set:** every single feature
has zero or *negative* importance, meaning shuffling that feature's values does not
hurt the Random Forest's held-out score, or actually helps it. That is a strong signal
the model has not learned a real, generalizable relationship between content/keyword
characteristics and `ctr_gap`, it has fit noise in the training clients that does not
transfer. `content_type` and `age_tier` show the most negative importance, consistent
with the model leaning hardest on exactly the categorical splits that do not
generalize.

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance

cat_features = ["content_type", "main_intent", "freshness_tier", "age_tier"]
num_features = ["search_volume", "competition", "cpc", "word_count"]

preprocess = ColumnTransformer([
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_features),
    ("num", SimpleImputer(strategy="median"), num_features),
])

X_train = train_df[cat_features + num_features]
y_train = train_df["ctr_gap"]
X_test = test_df[cat_features + num_features]
y_test = test_df["ctr_gap"]

results = [{"model": "Baseline rule (ML-07)", "r2": 0.0, "mae": float("nan")}]

dummy = DummyRegressor(strategy="mean").fit(X_train, y_train)
results.append({"model": "DummyRegressor (train mean)", "r2": r2_score(y_test, dummy.predict(X_test)),
                 "mae": mean_absolute_error(y_test, dummy.predict(X_test))})

lin_pipe = Pipeline([("prep", preprocess), ("model", LinearRegression())]).fit(X_train, y_train)
lin_pred = lin_pipe.predict(X_test)
results.append({"model": "Linear Regression", "r2": r2_score(y_test, lin_pred), "mae": mean_absolute_error(y_test, lin_pred)})

rf_pipe = Pipeline([("prep", preprocess),
                     ("model", RandomForestRegressor(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1))]).fit(X_train, y_train)
rf_pred = rf_pipe.predict(X_test)
results.append({"model": "Random Forest (depth=6)", "r2": r2_score(y_test, rf_pred), "mae": mean_absolute_error(y_test, rf_pred)})

comparison = pd.DataFrame(results)
print(f"Base rate (std of ctr_gap in test, for scale): {y_test.std():.4f}")
print()
print(comparison.round(4).to_string(index=False))
print()

perm = permutation_importance(rf_pipe, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({
    "feature": cat_features + num_features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
print("Permutation importance (Random Forest, held-out test set):")
print(imp_df.round(4).to_string(index=False))

Base rate (std of ctr_gap in test, for scale): 0.3268

                      model      r2    mae
      Baseline rule (ML-07)  0.0000    NaN
DummyRegressor (train mean) -0.0143 0.2192
          Linear Regression -0.5398 0.2713
    Random Forest (depth=6) -1.5655 0.3087

Permutation importance (Random Forest, held-out test set):
       feature  importance_mean  importance_std
 search_volume           0.0065          0.0020
   competition           0.0004          0.0002
           cpc          -0.0001          0.0002
   main_intent          -0.0324          0.0205
freshness_tier          -0.1004          0.0449
    word_count          -0.1226          0.0851
      age_tier          -0.1434          0.0091
  content_type          -0.1683          0.0457


## 4. Errors and interpretation

**Where the model is most wrong:** mean absolute residual by `content_type` on the
test set is 1.006 for `comparison article` (n=366), 0.678 for `feedly article`
(n=192), and only 0.202 for `keyword article` (n=3,056). The two minority content
types, the ones with the least training data, are where the model falls apart, it has
almost nothing to learn their pattern from.

**What it actually leans on, confirmed:** one training-only client,
`client_d4735e3a26` (61 rows), has a mean `ctr_gap` of 2.593, wildly out of line with
every other client (next highest is 0.841, most cluster between -0.26 and 0.13).
Retraining the Random Forest with that one client removed from training moves R^2 from
-1.5655 to -0.0702, nearly matching the trivial DummyRegressor's -0.0143. One client's
noise, not a real content signal, was driving most of the damage. This lines up with
Section 3's permutation importance, `content_type` and `age_tier` had the most negative
importance, exactly the categorical splits that outlier client would have pulled hardest
on.

**Three concrete wrong cases**, all three, unprompted, come from the same single test
client (`client_4ec9599fc2`):

1. `content_8606dd109b92`, feedly article, actual `ctr_gap` = 0.025, predicted = 4.766,
   off by 4.74. Wrong because the model output the same extreme number regardless of
   this row's actual (modest) outcome.
2. `content_0d3276c90637`, keyword article, actual `ctr_gap` = 4.605 (a real, unusually
   large gap), predicted = -0.076, off by 4.68 the other direction. The model cannot
   distinguish a page that IS a genuine extreme outlier from one that only looks like
   the memorized outlier client on paper.
3. `content_32bc200c6240`, feedly article, actual `ctr_gap` = 0.345, predicted = 4.766,
   again the same near-constant 4.77 prediction as case 1.

Two of these three get the identical ~4.77 prediction: this is not the model learning a
driver of CTR gap, it is the model reproducing one client's extreme training value for
any row that superficially resembles it on `content_type`/`age_tier`, then applying that
number blindly regardless of the row's real, current `search_volume` or `word_count`.

**Honest conclusion:** the content and keyword characteristics available in this
starter dataset do not meaningfully explain `ctr_gap`, once training-client noise is
accounted for, both models sit close to a trivial mean predictor at best. The Week-4
rule cannot explain *why* a page underperforms either, but it does not need to, it
never claimed to, it only measures the gap and ranks by volume. For now, the honest
recommendation is to keep using the rule for ranking and treat this week's model as a
real, informative negative: it tells us content/keyword metadata is not where the
explanation lives, not that no explanation exists.

In [14]:
test_df = test_df.copy()
test_df["predicted_ctr_gap"] = rf_pred
test_df["residual"] = test_df["ctr_gap"] - test_df["predicted_ctr_gap"]
test_df["abs_residual"] = test_df["residual"].abs()

print("Mean absolute residual by content_type:")
print(test_df.groupby("content_type")["abs_residual"].agg(["mean", "count"]).round(3))
print()

# Is one training client driving the damage? Check the most extreme train-client mean ctr_gap.
print("Train client mean ctr_gap, top 3 most extreme:")
print(train_df.groupby("client_id")["ctr_gap"].mean().sort_values(ascending=False).head(3).round(3))
print()

outlier_client = "client_d4735e3a26"
train_no_outlier = train_df[train_df["client_id"] != outlier_client]
X_train2 = train_no_outlier[cat_features + num_features]
y_train2 = train_no_outlier["ctr_gap"]
rf_pipe2 = Pipeline([("prep", preprocess),
                      ("model", RandomForestRegressor(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1))]).fit(X_train2, y_train2)
rf_pred2 = rf_pipe2.predict(X_test)
print(f"Random Forest R^2 on test, WITHOUT outlier client in training: {r2_score(y_test, rf_pred2):.4f}")
print(f"Random Forest MAE on test, WITHOUT outlier client in training: {mean_absolute_error(y_test, rf_pred2):.4f}")
print()

# Three worst-error rows
worst = test_df.sort_values("abs_residual", ascending=False).head(3)
cols = ["content_id", "client_id", "content_type", "freshness_tier", "age_tier",
        "word_count", "search_volume", "ctr_gap", "predicted_ctr_gap", "residual"]
print("Three worst-error rows:")
print(worst[cols].to_string(index=False))

Mean absolute residual by content_type:
                     mean  count
content_type                    
comparison article  1.006    366
feedly article      0.678    192
keyword article     0.202   3056

Train client mean ctr_gap, top 3 most extreme:
client_id
client_d4735e3a26    2.593
client_0b918943df    0.841
client_9f14025af0    0.577
Name: ctr_gap, dtype: float64

Random Forest R^2 on test, WITHOUT outlier client in training: -0.0702
Random Forest MAE on test, WITHOUT outlier client in training: 0.2248

Three worst-error rows:
          content_id         client_id    content_type freshness_tier age_tier  word_count  search_volume  ctr_gap  predicted_ctr_gap  residual
content_8606dd109b92 client_4ec9599fc2  feedly article           0-30  181-365      1113.0            NaN  0.02524           4.765720 -4.740479
content_0d3276c90637 client_4ec9599fc2 keyword article         91-180     365+         NaN            0.0  4.60524          -0.075864  4.681105
content_32bc200c6240 client

Self-check for ML-08:

All four sections filled with real reasoning and real numbers: ✅
Runs top to bottom clean: ✅, confirmed against your own three runs
No client names or private queries: ✅, only pseudonymized client_id/content_id
Careful words, no overclaiming: ✅, and this notebook does something harder than usual, it reports a real negative result honestly instead of tuning until a number looked good, which is exactly what training-honest-models asks for
Committed: (work/notebooks/w05_model.ipynb)